# Bulk ATAC-seq part 2: consensus peaks and MACS3 differential peaks

**Case study: How does chromatin accessibility in the liver change in a mouse model of diabetes and high-fat diet?**

Jeong BK, Choi WI, Choi W, et al. *A male mouse model for metabolic dysfunction-associated steatotic liver disease and hepatocellular carcinoma.* Nature Communications (2024) 15:6506. https://doi.org/10.1038/s41467-024-50660-y

**Research questions:**

1. How good are the ATAC-seq libraries, and how does their quality shape the results?
2. Which regions change accessibility in the livers of STZ + HFD mice, compared with control mice?
3. Which genes lie near these regions?

In this notebook we prepare the analysis of question 2 (consensus peaks and counts) and see a first, simple answer with MACS3; [part 3](part_3_diffbind_deseq2.ipynb) answers questions 2 and 3 with statistics that use the replicates.

## Introduction

In [part 1](part_1_reads_to_peaks.ipynb) we called peaks for two mice and saw that one mouse versus one mouse cannot tell us whether accessibility changes. Now we use all **12 mice** (4 groups × 3 replicates) of [GSE246213](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE246213).

To compare samples, we first need **one common set of regions** in which we count fragments for every sample. Peaks called per sample are not directly comparable: the same open region gets slightly different boundaries in each sample, and weak peaks are called in some samples but not in others. A **consensus peak set** solves this. There are several ways to build one, and the choice changes the regions you test, so we compare three:

1. the **union** of all peaks, merged where they overlap;
2. peaks **supported by at least *N* samples**;
3. **fixed-width peaks around summits**, selected iteratively by significance (the method of Corces et al. 2018, also used by ArchR and SnapATAC2).

We then **count fragments** per sample in the consensus peaks: this count matrix is the input for DiffBind and DESeq2 in [part 3](part_3_diffbind_deseq2.ipynb).

Finally, we try a peak caller's own differential mode, **MACS3 `bdgdiff`**, and see why it is not enough on its own.

All files are restricted to chr19 so everything runs in minutes; the steps are identical genome-wide.

## Step 0: Setup

This notebook uses the `atac-cli` environment and the "Bash (atac-cli)" kernel; see [part 1](part_1_reads_to_peaks.ipynb) for how to install them. Input data are in the folder `data` next to this notebook, our results go to `data/results/part_2`.

In [1]:
# The folder of this notebook (the kernel starts there). Kept when you run this cell again after the cd below.
HERE=${HERE:-$PWD}

# Input data, in the folder data next to this notebook
BULK=$HERE/data
GENOME=$BULK/genome/mm10

# Your own results
OUT=$BULK/results/part_2
mkdir -p $OUT && cd $OUT

THREADS=4
SAMPLES=$(tail -n +2 $BULK/samplesheet.tsv | cut -f1 | tr '\n' ' ')
echo "samples: $SAMPLES"

# effective size of chr19 (non-N bases), for MACS3
GSIZE=$(samtools faidx $GENOME/mm10.fa chr19 | grep -v '>' | tr -d 'Nn\n' | wc -c)
echo "chr19 effective size: $GSIZE"

samples: SCD_1 SCD_2 SCD_3 STZ_SCD_1 STZ_SCD_2 STZ_SCD_3 HFD_1 HFD_2 HFD_3 STZ_HFD_1 STZ_HFD_2 STZ_HFD_3 
chr19 effective size: 58205856


## Step 1: The filtered BAM files

The 12 samples were processed exactly like in [part 1](part_1_reads_to_peaks.ipynb) (trimming, alignment, duplicate marking, filtering) on the full data, with `scripts/10_bulk_pipeline.sh`. We link the chr19 part of each filtered BAM:

In [2]:
mkdir -p bam
# ln -sf: a symbolic link instead of a copy (saves disk space); we also need the index (.bai)
for s in $SAMPLES; do
    ln -sf $BULK/data_generated/$s/$s.filtered.chr19.bam bam/$s.bam
    ln -sf $BULK/data_generated/$s/$s.filtered.chr19.bam.bai bam/$s.bam.bai
done
# -f 64: count only the first read of each pair, i.e. fragments
for s in $SAMPLES; do
    printf "%-10s %9d fragments on chr19\n" $s $(samtools view -c -f 64 bam/$s.bam)
done

SCD_1         449050 fragments on chr19
SCD_2         406150 fragments on chr19
SCD_3         448687 fragments on chr19
STZ_SCD_1     436726 fragments on chr19
STZ_SCD_2     514927 fragments on chr19
STZ_SCD_3     526856 fragments on chr19
HFD_1         538826 fragments on chr19
HFD_2         509634 fragments on chr19
HFD_3         465873 fragments on chr19
STZ_HFD_1     295926 fragments on chr19
STZ_HFD_2     402781 fragments on chr19
STZ_HFD_3     460395 fragments on chr19


## Step 2: Peaks per sample

We call peaks for each sample with the same MACS3 settings as in part 1 (paired-end mode, q < 0.01). This takes a few minutes.

In [3]:
mkdir -p peaks
for s in $SAMPLES; do
    macs3 callpeak -f BAMPE -t bam/$s.bam -n $s -g $GSIZE --keep-dup all -q 0.01 \
        --outdir peaks 2> peaks/$s.log
done
wc -l peaks/*_peaks.narrowPeak

   356 peaks/HFD_1_peaks.narrowPeak
   384 peaks/HFD_2_peaks.narrowPeak
   368 peaks/HFD_3_peaks.narrowPeak
   478 peaks/SCD_1_peaks.narrowPeak
   438 peaks/SCD_2_peaks.narrowPeak
   900 peaks/SCD_3_peaks.narrowPeak
   126 peaks/STZ_HFD_1_peaks.narrowPeak
   272 peaks/STZ_HFD_2_peaks.narrowPeak
   389 peaks/STZ_HFD_3_peaks.narrowPeak
   389 peaks/STZ_SCD_1_peaks.narrowPeak
   465 peaks/STZ_SCD_2_peaks.narrowPeak
   389 peaks/STZ_SCD_3_peaks.narrowPeak
  4954 total


*If peak calling takes too long, use the pre-generated peaks: `cp -r $BULK/nb02_generated/peaks .`*

You should get between about 130 (STZ_HFD_1) and 900 (SCD_3) peaks per sample on chr19. The number of peaks follows the quality of each library (compare with the QC table of [part 1](part_1_reads_to_peaks.ipynb)) at least as much as the biology.

How similar are the peak sets? The **Jaccard index** of two sets of regions is the number of base pairs they share divided by the number of base pairs covered by either (0 = no overlap, 1 = identical). Let's compare a replicate pair with a pair from different groups:

In [4]:
jaccard() {  # jaccard index of the peaks of two samples
    # <( ... ): pass the output of a command as if it were a file; bedtools jaccard needs sorted input
    # the last line of the output holds the numbers; column 3 is the Jaccard index
    bedtools jaccard -a <(sort -k1,1 -k2,2n peaks/$1_peaks.narrowPeak) \
                     -b <(sort -k1,1 -k2,2n peaks/$2_peaks.narrowPeak) | tail -1 | cut -f3
}
echo "SCD_1 vs SCD_2 (replicates):         $(jaccard SCD_1 SCD_2)"
echo "SCD_1 vs STZ_HFD_1 (different group): $(jaccard SCD_1 STZ_HFD_1)"

SCD_1 vs SCD_2 (replicates):         0.502851
SCD_1 vs STZ_HFD_1 (different group): 0.177116


**Question**: Are replicates of the same group more similar to each other than to other groups? How could the number of peaks per sample (and so library quality) affect the Jaccard index? (Exercise 1 at the end of this notebook compares all pairs of samples.)

## Step 3: Consensus peaks

### 3a. Union of all peaks

The simplest consensus: pool the peaks of all samples and merge everything that overlaps (`bedtools merge`).

In [6]:
# pool the peaks of all samples, keep chr/start/end, sort by position (bedtools merge needs sorted input) and merge overlaps
cat peaks/*_peaks.narrowPeak | cut -f1-3 | sort -k1,1 -k2,2n | bedtools merge > consensus_union.bed

widths() {  # summary of region widths in a BED file
    awk '{print $3 - $2}' $1 | sort -n | awk '{w[NR] = $1; s += $1}
        END {printf "%6d regions, width min %d / median %d / max %d bp, %.1f Mb in total\n", NR, w[1], w[int(NR/2)+1], w[NR], s/1e6}'
}
echo "per-sample peaks (all 12 pooled): $(cat peaks/*_peaks.narrowPeak | wc -l)"
echo -n "union: "; widths consensus_union.bed

per-sample peaks (all 12 pooled): 4954
union:   1178 regions, width min 140 / median 357 / max 2082 bp, 0.5 Mb in total


Look at the maximum width. Merging "daisy-chains" peaks: if sample A has a peak that overlaps one of sample B, which overlaps one of sample C, all three become one long region. A broad region can hide a change in one of its parts. The union also keeps every peak that was called in only one sample, which is often noise.

### 3b. Peaks supported by at least *N* samples

We can require that a merged region overlaps peaks from at least *N* different samples. This is what DiffBind does by default (`minOverlap = 2`). `bedtools merge -c 4 -o count_distinct` counts the distinct sample names in each merged region:

In [7]:
# add the sample name as column 4 to every peak, pool and merge; -c 4 -o count_distinct: number of different samples per region
for s in $SAMPLES; do
    awk -v s=$s 'BEGIN {OFS = "\t"} {print $1, $2, $3, s}' peaks/${s}_peaks.narrowPeak
done | sort -k1,1 -k2,2n | bedtools merge -c 4 -o count_distinct > union_with_support.bed

echo "N  regions supported by >= N samples"
for n in 1 2 3 4 6 9 12; do
    printf "%-2d %d\n" $n $(awk -v n=$n '$4 >= n' union_with_support.bed | wc -l)
done
awk '$4 >= 2' union_with_support.bed | cut -f1-3 > consensus_min2.bed
echo -n "min 2 samples: "; widths consensus_min2.bed

N  regions supported by >= N samples
1  1178
2  659
3  526
4  444
6  353
9  251
12 85
min 2 samples:    659 regions, width min 163 / median 589 / max 2082 bp, 0.4 Mb in total


**Question**: Why should *N* not be larger than 3 in this experiment? (Hint: think of a region that opens in only one of the four groups.)

### 3c. Fixed-width peaks around summits, selected iteratively

The third strategy avoids both long merged regions and arbitrary boundaries:

1. take the **summit** of every peak in every sample (the position with the highest pileup, `*_summits.bed`) and extend it to a fixed **500 bp** window;
2. give each window a score that is comparable between samples: its -log10 q-value divided by the sum over all summits of that sample, times 10<sup>6</sup> ("score per million");
3. keep the highest-scoring window, remove all windows that overlap it, and repeat until no windows are left.

The result is a set of non-overlapping 500 bp regions, each centred on the strongest summit in its neighbourhood. In bash, step 3 can be done in rounds: `bedtools cluster` groups overlapping windows, we keep the best of each group, and remove everything that overlaps a kept window.

In [8]:
# 1-2. 500 bp windows around all summits, with a normalised score (column 5 of *_summits.bed is -log10 q)
for s in $SAMPLES; do
    awk -v s=$s 'BEGIN {OFS = "\t"} {line[NR] = $0; total += $5}
        END {for (i = 1; i <= NR; i++) {
                 split(line[i], f, "\t"); start = f[2] - 250; if (start < 0) start = 0
                 print f[1], start, f[2] + 250, s "_" i, f[5] / total * 1e6 }}' peaks/${s}_summits.bed
done | sort -k1,1 -k2,2n > summit_windows.bed
echo "$(wc -l < summit_windows.bed) summit windows"

# 3. iterative selection
cp summit_windows.bed remaining.bed; : > selected.bed; round=0
while [ -s remaining.bed ]; do
    round=$(( round + 1 ))
    bedtools cluster -i remaining.bed | sort -k6,6n -k5,5gr | awk '!seen[$6]++' | cut -f1-5 \
        | sort -k1,1 -k2,2n > best.bed
    cat best.bed >> selected.bed
    bedtools intersect -v -a remaining.bed -b best.bed > tmp.bed && mv tmp.bed remaining.bed
    echo "round $round: kept $(wc -l < best.bed), $(wc -l < remaining.bed) windows left"
done
sort -k1,1 -k2,2n selected.bed > consensus_fixed500_all.bed
rm remaining.bed best.bed selected.bed
echo -n "fixed width: "; widths consensus_fixed500_all.bed

4954 summit windows
round 1: kept 1154, 99 windows left
round 2: kept 52, 2 windows left
round 3: kept 2, 0 windows left
fixed width:   1208 regions, width min 500 / median 500 / max 500 bp, 0.6 Mb in total


Like the union, this set still contains windows whose summit was found in a single sample. We combine it with the idea of 3b: keep only windows that overlap peaks of **at least 2 samples**.

In [9]:
cut -f1-4 consensus_fixed500_all.bed > windows.tmp
# for each sample a 0/1 column: does the window overlap a peak of that sample? (intersect -c counts the overlaps)
for s in $SAMPLES; do
    bedtools intersect -c -a windows.tmp -b peaks/${s}_peaks.narrowPeak | awk '{print ($5 > 0)}' > support_$s.tmp
done
paste windows.tmp support_*.tmp | awk 'BEGIN {OFS = "\t"} {n = 0; for (i = 5; i <= NF; i++) n += $i; if (n >= 2) print $1, $2, $3, "peak" NR}' \
    > consensus_peaks.bed
rm windows.tmp support_*.tmp
echo -n "fixed width, >= 2 samples: "; widths consensus_peaks.bed

fixed width, >= 2 samples:    715 regions, width min 500 / median 500 / max 500 bp, 0.4 Mb in total


### Comparing the three consensus sets

Load `consensus_union.bed`, `consensus_min2.bed` and `consensus_peaks.bed` in IGV together with a few per-sample peak files (go to *Scd1* or *Cpt1a* again). How do the regions differ?

Here is the overview:

In [10]:
for f in consensus_union.bed consensus_min2.bed consensus_fixed500_all.bed consensus_peaks.bed; do
    printf "%-28s" $f; widths $f
done

consensus_union.bed           1178 regions, width min 140 / median 357 / max 2082 bp, 0.5 Mb in total
consensus_min2.bed             659 regions, width min 163 / median 589 / max 2082 bp, 0.4 Mb in total
consensus_fixed500_all.bed    1208 regions, width min 500 / median 500 / max 500 bp, 0.6 Mb in total
consensus_peaks.bed            715 regions, width min 500 / median 500 / max 500 bp, 0.4 Mb in total


There is no single correct choice, but for differential analysis the fixed-width set has clear advantages: every region has the same size (so counts are comparable between regions), regions are centred on the summit (where the change usually is), and there is no daisy-chaining. It is also the standard input for downstream tools such as motif enrichment or sequence-based models. We continue with `consensus_peaks.bed`.

## Step 4: Counting fragments in the consensus peaks

[featureCounts](https://subread.sourceforge.net/featureCounts.html) (from the Subread package) counts reads or fragments per region. It reads regions in **SAF** format (GeneID, Chr, Start, End, Strand; 1-based coordinates):

In [11]:
# BED is 0-based (start not included), SAF is 1-based: add 1 to the start
awk 'BEGIN {OFS = "\t"; print "GeneID", "Chr", "Start", "End", "Strand"} {print $4, $1, $2 + 1, $3, "."}' \
    consensus_peaks.bed > consensus_peaks.saf
head -3 consensus_peaks.saf

GeneID	Chr	Start	End	Strand
peak1	chr19	3282714	3283213	.
peak2	chr19	3323164	3323663	.


Options:

- `-p --countReadPairs`: count **fragments** (read pairs), not single reads;
- `-O`: a fragment overlapping two neighbouring windows is counted in both (otherwise it is discarded as "ambiguous");
- `-F SAF -a`: the regions.

In [12]:
# -T: threads; -o: output table (+ a .summary file); the BAM files of all samples follow
featureCounts -T $THREADS -p --countReadPairs -O -F SAF -a consensus_peaks.saf \
    -o featurecounts.txt $(for s in $SAMPLES; do echo bam/$s.bam; done) 2> featurecounts.log
tail -4 featurecounts.log
# a clean matrix: region id + one column per sample (featureCounts writes 6 annotation columns before the counts)
( printf "peak"; for s in $SAMPLES; do printf "\t%s" $s; done; echo
  tail -n +3 featurecounts.txt | cut -f1,7- ) > counts_matrix.tsv
head -5 counts_matrix.tsv | column -t

|| ry"                                                                        ||
||                                                                            ||
\\============================================================================//

peak   SCD_1  SCD_2  SCD_3  STZ_SCD_1  STZ_SCD_2  STZ_SCD_3  HFD_1  HFD_2  HFD_3  STZ_HFD_1  STZ_HFD_2  STZ_HFD_3
peak1  33     35     54     39         28         32         25     19     24     19         22         44
peak2  68     65     79     64         92         61         49     80     61     36         48         42
peak5  43     39     63     34         43         23         28     21     39     9          21         23
peak7  64     76     76     79         68         62         60     63     60     32         43         56


The `.summary` file tells how many fragments were assigned to a region. Assigned / total is the **FRiP** of each sample in the consensus set. Let's compute it for every sample: are they similar?

In [13]:
# the .summary table has one row per category (Assigned, Unassigned_NoFeatures, ...) and one column per sample:
# FRiP = Assigned / sum of all categories
awk -v samples="$SAMPLES" 'BEGIN {n = split(samples, name, " ")}
    NR > 1 {for (i = 2; i <= NF; i++) {total[i] += $i; if ($1 == "Assigned") assigned[i] = $i}}
    END {for (i = 2; i <= n + 1; i++) printf "%-10s FRiP %.3f\n", name[i-1], assigned[i] / total[i]}' featurecounts.txt.summary

SCD_1      FRiP 0.071
SCD_2      FRiP 0.080
SCD_3      FRiP 0.104
STZ_SCD_1  FRiP 0.083
STZ_SCD_2  FRiP 0.081
STZ_SCD_3  FRiP 0.073
HFD_1      FRiP 0.065
HFD_2      FRiP 0.067
HFD_3      FRiP 0.069
STZ_HFD_1  FRiP 0.048
STZ_HFD_2  FRiP 0.061
STZ_HFD_3  FRiP 0.066


FRiP in the consensus peaks ranges from about 0.05 (STZ_HFD_1) to 0.10 (SCD_3), and the SCD samples are on average higher than the STZ_HFD samples. When FRiP differs, normalising by the total number of fragments (library size) and normalising by the fragments *in peaks* give different results: a sample with a higher FRiP looks "more accessible everywhere" after library-size normalisation. We will compare both normalisations in [part 3](part_3_diffbind_deseq2.ipynb).

## Step 5: MACS3 differential peaks (`bdgdiff`)

MACS3 can compare two conditions directly. `macs3 bdgdiff` takes, for each condition, the **pileup** of fragments and the **local background (lambda)** as bedGraph files (written by `callpeak -B`), and reports regions where one condition is significantly higher than the other, using a log-likelihood ratio (LLR).

It works on one track per condition, so we **pool** the three replicates of each group into a single `callpeak` run. We compare the two most different groups: STZ + HFD versus standard chow (SCD).

In [14]:
mkdir -p pooled
for g in SCD STZ_HFD; do
    # three BAM files after -t are pooled into one sample; -B also writes the pileup and lambda bedGraphs for bdgdiff
    macs3 callpeak -f BAMPE -t bam/${g}_1.bam bam/${g}_2.bam bam/${g}_3.bam -n $g -g $GSIZE \
        --keep-dup all -q 0.01 -B --outdir pooled 2> pooled/$g.log
done
ls -lh pooled/*.bdg

-rw-r--r-- 1 nicolo.casonato sg_c20_pi_stein_aerts 62M Oct 28 01:23 pooled/SCD_control_lambda.bdg
-rw-r--r-- 1 nicolo.casonato sg_c20_pi_stein_aerts 71M Oct 28 01:23 pooled/SCD_treat_pileup.bdg
-rw-r--r-- 1 nicolo.casonato sg_c20_pi_stein_aerts 68M Oct 28 01:24 pooled/STZ_HFD_control_lambda.bdg
-rw-r--r-- 1 nicolo.casonato sg_c20_pi_stein_aerts 65M Oct 28 01:24 pooled/STZ_HFD_treat_pileup.bdg


`bdgdiff` needs the sequencing depth of each condition (`--d1`, `--d2`) to scale the two tracks to each other. MACS3 writes it in the header of `*_peaks.xls`; instead of copying the numbers by hand, we read them from the file:

In [15]:
depth() {  # number of fragments used by callpeak, from the .xls header
    # "after filtering" is only written when MACS3 removed duplicates; with --keep-dup all the total is used
    awk -F': ' '/^# (fragments|tags) after filtering in treatment/ {f = $2}
                /^# total (fragments|tags) in treatment/ {t = $2}
                END {print (f ? f : t)}' $1
}
D_STZ_HFD=$(depth pooled/STZ_HFD_peaks.xls)
D_SCD=$(depth pooled/SCD_peaks.xls)
echo "STZ_HFD: $D_STZ_HFD fragments, SCD: $D_SCD fragments"

STZ_HFD: 1159102 fragments, SCD: 1303887 fragments


Other options: `-l` is the minimum length of a differential region, `-g` the maximum gap to merge nearby regions, `-C` the LLR cutoff (default 3, i.e. the difference is 10<sup>3</sup> = 1000 times more likely than no difference). bdgdiff writes three files: regions higher in condition 1 (`cond1`), higher in condition 2 (`cond2`), and similar in both (`common`).

In [16]:
mkdir -p bdgdiff
# condition 1 = STZ_HFD, condition 2 = SCD: --t = treatment pileup, --c = control lambda (local background), --d = depth
macs3 bdgdiff \
    --t1 pooled/STZ_HFD_treat_pileup.bdg --c1 pooled/STZ_HFD_control_lambda.bdg \
    --t2 pooled/SCD_treat_pileup.bdg     --c2 pooled/SCD_control_lambda.bdg \
    --d1 $D_STZ_HFD --d2 $D_SCD -l 120 -g 60 -C 3 \
    --outdir bdgdiff --o-prefix STZ_HFD_vs_SCD 2> bdgdiff/STZ_HFD_vs_SCD.log
wc -l bdgdiff/STZ_HFD_vs_SCD_*.bed

  537 bdgdiff/STZ_HFD_vs_SCD_c3.0_common.bed
   35 bdgdiff/STZ_HFD_vs_SCD_c3.0_cond1.bed
  519 bdgdiff/STZ_HFD_vs_SCD_c3.0_cond2.bed
 1091 total


Each file has a header line, so the number of regions is one less than the line count. The strongest differences (column 5 is the LLR score):

In [17]:
# tail -n +2: skip the header line; sort on column 5 (the LLR score), highest first
echo "== more accessible in STZ_HFD"; tail -n +2 bdgdiff/STZ_HFD_vs_SCD_c3.0_cond1.bed | sort -k5,5gr | head -5
echo "== more accessible in SCD";     tail -n +2 bdgdiff/STZ_HFD_vs_SCD_c3.0_cond2.bed | sort -k5,5gr | head -5

== more accessible in STZ_HFD
chr19	50313290	50313473	STZ_HFD_vs_SCD_cond1_31	8.96629
chr19	50407573	50407738	STZ_HFD_vs_SCD_cond1_32	8.68387
chr19	32224552	32224837	STZ_HFD_vs_SCD_cond1_20	8.61423
chr19	40999054	40999188	STZ_HFD_vs_SCD_cond1_28	8.37313
chr19	29419739	29420068	STZ_HFD_vs_SCD_cond1_18	8.21203
== more accessible in SCD
chr19	37455953	37456226	STZ_HFD_vs_SCD_cond2_302	28.0667
chr19	4024262	4024474	STZ_HFD_vs_SCD_cond2_26	14.0649
chr19	6363254	6364155	STZ_HFD_vs_SCD_cond2_132	13.4887
chr19	4121212	4121668	STZ_HFD_vs_SCD_cond2_31	13.2721
chr19	5831615	5831786	STZ_HFD_vs_SCD_cond2_101	13.1009


### How reliable is this?

Pooling throws away the information that makes statistics possible: **how much do mice of the same group differ?** A simple test of any differential method is to compare two mice **of the same group**: any region it reports is a false positive.

Let's run `callpeak -B` on two control mice, `SCD_2` and `SCD_3`, separately and compare them with `bdgdiff`, using the same settings, and put the numbers next to those of STZ_HFD vs SCD:

In [18]:
mkdir -p single
# call peaks on each mouse separately, again with -B to write the pileup and lambda tracks
for s in SCD_2 SCD_3; do
    macs3 callpeak -f BAMPE -t bam/$s.bam -n $s -g $GSIZE --keep-dup all -q 0.01 -B --outdir single 2> single/$s.log
done
# the same bdgdiff settings as above; the depths again come from the .xls headers
macs3 bdgdiff \
    --t1 single/SCD_2_treat_pileup.bdg --c1 single/SCD_2_control_lambda.bdg \
    --t2 single/SCD_3_treat_pileup.bdg --c2 single/SCD_3_control_lambda.bdg \
    --d1 $(depth single/SCD_2_peaks.xls) --d2 $(depth single/SCD_3_peaks.xls) -l 120 -g 60 -C 3 \
    --outdir bdgdiff --o-prefix SCD_2_vs_SCD_3 2> bdgdiff/SCD_2_vs_SCD_3.log
# number of regions per file (minus the header line)
for f in bdgdiff/SCD_2_vs_SCD_3_c3.0_cond?.bed bdgdiff/STZ_HFD_vs_SCD_c3.0_cond?.bed; do
    printf "%-45s %6d regions\n" $f $(( $(wc -l < $f) - 1 ))
done

bdgdiff/SCD_2_vs_SCD_3_c3.0_cond1.bed              9 regions
bdgdiff/SCD_2_vs_SCD_3_c3.0_cond2.bed            172 regions
bdgdiff/STZ_HFD_vs_SCD_c3.0_cond1.bed             34 regions
bdgdiff/STZ_HFD_vs_SCD_c3.0_cond2.bed            518 regions


Two mice that received exactly the same treatment already give about 180 "differential" regions, a third of the ~550 of STZ_HFD vs SCD. Notice also the direction: almost all of them are "higher" in SCD_3, the control library with the highest FRiP, just like almost all STZ_HFD vs SCD regions are "higher" in SCD. Differences in library quality look like differential accessibility. Many of the STZ_HFD vs SCD regions are therefore likely to be mouse-to-mouse or library-to-library variation rather than treatment effects, and `bdgdiff` gives us no way to tell them apart: it has no estimate of the biological variance and no false discovery rate across regions.

`bdgdiff` is still useful to explore a comparison quickly, or when you really have no replicates; the [ChIP-seq practical](../chip-seq/index.md) uses it that way, to compare H3K27ac between one liver and one spleen sample. With replicates, use a method that models the variation between replicates: that is what **DiffBind** and **DESeq2** do in [part 3](part_3_diffbind_deseq2.ipynb), starting from the count matrix we just built.

## Summary

| file | content | used in |
|---|---|---|
| `consensus_peaks.bed` / `.saf` | 500 bp consensus peaks, supported by ≥ 2 samples | part 3 |
| `counts_matrix.tsv` | fragments per consensus peak (rows) and sample (columns) | part 3 (DESeq2) |
| `peaks/*_peaks.narrowPeak` | per-sample peaks | part 3 (DiffBind) |
| `bdgdiff/STZ_HFD_vs_SCD_*.bed` | MACS3 differential regions | part 3 (comparison) |

## Exercises

1. Compute the Jaccard index for all pairs of samples and print it as a 12 × 12 table (use the `jaccard` function of Step 2 in two nested loops). Are replicates of the same group more similar to each other than to other groups? How does the Jaccard index relate to the number of peaks per sample?
2. Build a consensus with peaks supported by at least 3 samples instead of 2 (Step 3b and the end of 3c). How many regions do you get, and how does the FRiP of each sample change?
3. Compare two other mice of the same group with `bdgdiff` (e.g. `HFD_1` and `HFD_2`). How many "differential" regions do you find, and in which direction? Can you predict the direction from the quality table of [part 1](part_1_reads_to_peaks.ipynb)?